# Online amplitude-encoding tutorial

This notebook accompanies the paper and shows how to generate the observables used in the online amplitude-encoding quantum reservoir computing protocol. The workflow is organized into three layers:

1. an ideal density-matrix reference model;
2. a gate-based circuit simulation with mid-circuit measurements and reset;
3. a hardware run on IBM Quantum using the same circuit structure.

The notebook is intended as a tutorial. The heavier figure-generation and post-processing steps are kept in `reproduce_figures.ipynb`.


## 1. Imports and local utilities

The shared Hilbert-space and density-matrix helpers are imported from `utilities.py`. The remaining imports are Qiskit tools used for circuit construction, simulation, transpilation, and hardware execution.


In [1]:
# Numerical and plotting stack
import sys
import json
import functools

import matplotlib
import numpy as np
import qiskit
import scipy.linalg
from tqdm import tqdm

# Qiskit backends and transpilation utilities
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import RZZGate
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

# Local utilities shared with the companion notebooks
from utilities import CompositeHilbertSpace, DensityMatrix, HilbertSpace

print("qiskit", qiskit.__version__)
print("python", sys.version.split()[0])
print("numpy", np.__version__)
print("matplotlib", matplotlib.__version__)

qiskit 1.4.2
python 3.11.11
numpy 2.0.2
matplotlib 3.10.1


## 2. Hilbert-space and Hamiltonian helpers

The reservoir contains `R` input/readout qubits and `M` memory qubits. This notebook keeps two TFIM variants because they serve different purposes in the paper:

1. **All-to-all spin model**: used for the ideal benchmark comparison with the original spin-network proposal, as in Fig. 2 of the paper. In the notebook this is provided by `get_hamiltonian_all_to_all` and `get_exact_unitary_all_to_all`. These functions return a dense Hamiltonian/unitary and are intended for density-matrix simulations, not for hardware execution.

2. **Nearest-neighbor hardware-tailored model**: used for the circuit and hardware data generation. In the notebook this is provided by `get_hamiltonian_nn`, `get_exact_unitary_nn`, and `get_trotter_step_nn`. The exact nearest-neighbor unitary is used as the ideal reference for the same model, while the Trotter step is the circuit block executed by the sampler/backend.

In short: use the all-to-all exact unitary to reproduce the Fig. 2 model comparison, and use the nearest-neighbor exact unitary plus Trotter step for the hardware-oriented workflow.


In [2]:
def initialize_space(M, R):
    """Build the composite Hilbert space for input/readout and memory qubits."""
    spin_space_M = HilbertSpace(M, "spin")
    spin_space_R = HilbertSpace(R, "spin")
    return CompositeHilbertSpace({"R": spin_space_R, "M": spin_space_M})

In [ ]:
def get_trotter_step_nn(M, R, h_0, J_bare, delta_t):
    """Build one first-order Trotter circuit step for the nearest-neighbor TFIM chain."""
    trotter_step = QuantumCircuit(M + R)

    # Transverse-field terms.
    for i in range(M + R):
        trotter_step.rz(h_0 * delta_t, i)

    # Even and odd nearest-neighbor XX layers.
    for i in range(0, M + R - 1, 2):
        trotter_step.rxx(2 * J_bare[i] * delta_t, i, i + 1)
    for i in range(1, M + R - 1, 2):
        trotter_step.rxx(2 * J_bare[i] * delta_t, i, i + 1)

    return trotter_step


def get_hamiltonian_all_to_all(M, R, h_0, J_s, seed=None):
    """Return the all-to-all transverse-field Ising Hamiltonian.

    This is the spin-network model used for the ideal benchmark comparison in
    Fig. 2 of the paper. The hardware-tailored experiment below instead uses
    the nearest-neighbor version to avoid SWAP overhead on the device.
    """
    if seed is not None:
        np.random.seed(seed)

    composite_space = initialize_space(M, R)
    hamiltonian = np.zeros((composite_space.dimension, composite_space.dimension))
    n_spins = M + R
    J_bare = np.random.uniform(-0.5, 0.5, (n_spins, n_spins))

    # All-to-all XX couplings.
    for i in range(1, n_spins + 1):
        for j in range(i + 1, n_spins + 1):
            hamiltonian += J_s * J_bare[i - 1, j - 1] * composite_space.X[i] @ composite_space.X[j]

    # Homogeneous transverse field.
    for i in range(1, n_spins + 1):
        hamiltonian += 0.5 * h_0 * composite_space.Z[i]

    return hamiltonian


def get_exact_unitary_all_to_all(M, R, h_0, J_s, delta_t, seed=None):
    """Return the exact dense unitary for the all-to-all spin-network benchmark.

    This unitary is useful for the Fig. 2-style comparison with the original
    spin model. It is not decomposed into hardware gates in this tutorial.
    """
    hamiltonian = get_hamiltonian_all_to_all(M, R, h_0, J_s, seed=seed)
    return scipy.linalg.expm(-1j * hamiltonian * delta_t)


def get_hamiltonian_nn(M, R, h_0, J_bare):
    """Return the nearest-neighbor transverse-field Ising Hamiltonian.

    This Hamiltonian defines the hardware-tailored model used below.
    """
    composite_space = initialize_space(M, R)
    hamiltonian = np.zeros((composite_space.dimension, composite_space.dimension))
    n_spins = M + R

    for i in range(1, n_spins):
        hamiltonian += J_bare[i - 1] * composite_space.X[i] @ composite_space.X[i + 1]

    for i in range(1, n_spins + 1):
        hamiltonian += 0.5 * h_0 * composite_space.Z[i]

    return hamiltonian


def get_unitary_and_trotter_step(M, R, h_0, J_s, delta_t, trotter_reps=10, seed=None):
    """Return the exact unitary and one Trotter step for the hardware-tailored model."""
    if seed is not None:
        np.random.seed(seed)

    n_spins = M + R
    J_bare = np.random.uniform(-0.5, 0.5, n_spins) * J_s

    hamiltonian = get_hamiltonian_nn(M, R, h_0, J_bare)
    U = scipy.linalg.expm(-1j * hamiltonian * delta_t)
    trotter_step = get_trotter_step_nn(M, R, h_0, J_bare, delta_t / trotter_reps)

    return U, trotter_step

## 3. Measurement and readout helpers

These functions convert bitstrings into expectation values, implement the weak-measurement backaction used by the density-matrix model, and define the basis rotations for measuring `X`, `Y`, and `Z`.

In [ ]:
def exp_z_from_counts(counts, bit_index, shots):
    """Estimate <Z> for one classical bit from Qiskit counts."""
    exp = 0.0
    for bitstring, count in counts.items():
        # Reverse the bitstring so that index 0 corresponds to classical bit c0.
        bit = int(bitstring[::-1][bit_index])
        z_value = 1 if bit == 0 else -1
        exp += z_value * count
    return exp / shots


In [ ]:
def M_matrix(N, g):
    """Return the element-wise backaction matrix induced by weak measurement."""
    single_qubit = np.array(
        [[1.0, np.exp(-g**2 / 2)], [np.exp(-g**2 / 2), 1.0]],
        dtype=np.complex128,
    )
    matrix = single_qubit.copy()
    for _ in range(N - 1):
        matrix = np.kron(matrix, single_qubit)
    return matrix


In [ ]:
def get_alpha(g):
    """Convert the continuous measurement strength g to the circuit parameter alpha."""
    val = 1 - np.exp(-g**2)
    if val < 0 or val > 1:
        raise ValueError("g is outside the valid range for alpha computation.")

    alpha = np.sqrt((1 + np.sqrt(val)) / 2)
    if alpha < 1 / np.sqrt(2) or alpha > 1:
        raise ValueError("Computed alpha is outside the physical interval [1/sqrt(2), 1].")

    return alpha


In [ ]:
def embed_matrix(matrix, N, i, to_multiply=False):
    """Embed a single-qubit matrix into an N-qubit tensor-product space."""
    if i < 0 or i >= N:
        raise ValueError("Index i is out of bounds for embedding.")
    if N == 1:
        return matrix

    if not to_multiply:
        op_list = [np.eye(2, dtype=complex) for _ in range(N)]
    else:
        # Used for Hadamard-product backaction matrices rather than operators.
        op_list = [np.ones((2, 2), dtype=complex) for _ in range(N)]

    op_list[i] = matrix
    return functools.reduce(np.kron, op_list)


In [ ]:
# Basis changes applied before a computational-basis measurement.
H = (1 / np.sqrt(2)) * np.array([[1, 1], [1, -1]], dtype=complex)
S = np.array([[1, 0], [0, -np.exp(1j * np.pi / 2)]], dtype=complex)

measurement_rotations = {
    "X": H,
    "Y": H @ S,
    "Z": np.eye(2, dtype=complex),
}


## 4. Ideal density-matrix evolution

The ideal model implements the amplitude-encoding update explicitly: the input/readout subsystem is reset according to the current scalar input, while the memory subsystem is carried forward through its reduced density matrix. The average backaction of indirect measurements is applied to the monitored memory qubits.


In [ ]:
def run_ideal_window(input_series, U):
    """Run the density-matrix reference model for one input window."""
    measured_qubits = [0] + list(range(1, M + R, 2))
    exp_values = {i: {obs: [] for obs in observables} for i in range(len(measured_qubits))}

    for obs in observables:
        rho = np.zeros((2**(M + R), 2**(M + R)), dtype=np.complex128)
        rho[0, 0] = 1.0

        for s_k in input_series:
            # Amplitude-encoding update: replace the input/readout qubit and
            # keep the reduced memory state.
            rho_M = composite_space.get_reduced_density_matrix(rho, "M")
            rho_R = np.array(
                [[1.0 - s_k, np.sqrt((1.0 - s_k) * s_k)],
                 [np.sqrt((1.0 - s_k) * s_k), s_k]],
                dtype=np.complex128,
            )
            rho = np.kron(rho_R, rho_M)
            rho = U @ rho @ U.conj().T

            # Record the requested single-qubit observables.
            for j, i in enumerate(measured_qubits):
                operator = getattr(composite_space, obs)[i + 1]
                ev = DensityMatrix(rho).get_expectation_value(operator)
                exp_values[j][obs].append(ev)

            # Apply the average backaction associated with indirect measurement
            # on the monitored memory qubits.
            for j in range(1, M + R, 2):
                rotation = embed_matrix(measurement_rotations[obs], M + R, j)
                rho = rotation @ rho @ rotation.conj().T

                backaction_matrix_full = embed_matrix(backaction_matrix, M + R, j, to_multiply=True)
                rho = np.multiply(backaction_matrix_full, rho)

                rho = rotation.conj().T @ rho @ rotation

    return exp_values


## 5. Circuit construction and simulation

The circuit version reproduces the same update rule using mid-circuit measurements and reset on the input/readout qubit. Memory observables are accessed through ancilla-assisted indirect measurements, and the ancilla outcomes are rescaled by the corresponding measurement-strength factor.


In [ ]:
def build_circuit(input_series, trotter_step):
    """Build one circuit per measurement basis for a complete input window."""
    n_qubits = R + M + M // 2 + (1 if M % 2 == 1 else 0)
    n_meas_per_step = 1 + M // 2 + (1 if M % 2 == 1 else 0)
    n_clbits = len(input_series) * n_meas_per_step
    corresponding_qubits = range(1, M + R, 2)

    circuits = []
    for obs in observables:
        qc = QuantumCircuit(n_qubits, n_clbits)

        for k, input_value in enumerate(input_series):
            base = k * n_meas_per_step

            # Input injection by amplitude encoding on the reset qubit.
            angle = 2 * np.arccos(np.sqrt(1 - input_value))
            qc.ry(angle, 0)

            # Hardware-tailored reservoir evolution.
            for _ in range(trotter_reps):
                qc.compose(trotter_step, range(M + R), inplace=True)

            # Rotate the input/readout qubit into the requested measurement basis.
            if obs == "X":
                qc.h(0)
            elif obs == "Y":
                qc.sdg(0)
                qc.h(0)

            qc.measure(0, base)
            qc.reset(0)

            # Indirectly measure selected memory qubits using ancilla probes.
            for idx, ancilla in enumerate(range(M + R, n_qubits)):
                memory_qubit = corresponding_qubits[idx]
                ancilla_index = base + 1 + (ancilla - (M + R))

                qc.ry(2 * np.arccos(alpha), ancilla)

                if obs == "X":
                    qc.h(memory_qubit)
                elif obs == "Y":
                    qc.sdg(memory_qubit)
                    qc.h(memory_qubit)

                qc.cx(memory_qubit, ancilla)
                qc.measure(ancilla, ancilla_index)
                qc.reset(ancilla)

                # Undo the basis rotation on the memory qubit.
                if obs == "X":
                    qc.h(memory_qubit)
                elif obs == "Y":
                    qc.h(memory_qubit)
                    qc.s(memory_qubit)

        circuits.append(qc)

    return circuits


In [ ]:
def run_circuit_sim(input_series, circuits):
    """Run the circuit model with the Aer sampler and return expectation values."""
    optimized_qc = transpile(circuits, optimization_level=3)
    sampler = AerSampler()
    shots = int(1e4)

    job = sampler.run(optimized_qc, shots=shots)
    result = job.result()

    all_expvals = {}
    for obs_idx, obs in enumerate(observables):
        counts = result[obs_idx].data.c.get_counts()
        for k in range(len(input_series)):
            base = k * n_meas_per_step

            lab0 = f"{obs}_0_step_{k}"
            all_expvals[lab0] = exp_z_from_counts(counts, base, shots)

            for j in range(M + R, n_qubits):
                anc_index = base + 1 + (j - (M + R))
                labj = f"{obs}_{j}_step_{k}"
                all_expvals[labj] = exp_z_from_counts(counts, anc_index, shots)

    exp_values = {i: {obs: [] for obs in observables} for i in range(len([0]) + len(list(range(1, M + R, 2))))}

    # Ancilla signals are rescaled to estimate the corresponding memory-qubit observables.
    for obs in observables:
        for k in range(len(input_series)):
            for j in range(M + R, n_qubits):
                labj = f"{obs}_{j}_step_{k}"
                exp_values[j - (M + R) + 1][obs].append(all_expvals[labj] / (2 * alpha**2 - 1))

    for obs in observables:
        for k in range(len(input_series)):
            lab0 = f"{obs}_0_step_{k}"
            exp_values[0][obs].append(all_expvals[lab0])

    return exp_values


## 6. Hardware configuration

The following cells configure the IBM Quantum backend used in the paper. Replace the placeholder credentials before running them. The initial layout encodes the logical connectivity pattern used to keep the reservoir chain and memory-ancilla pairs local on the device.


In [ ]:
# Configure access to IBM Quantum.
# Replace the placeholder token/instance with your own credentials before running hardware jobs.
service = QiskitRuntimeService(
    channel="ibm_cloud",
    token="your_token_here",
    instance="cnr",
)


In [ ]:
def fix_rzz_angles_postprocess(circ: QuantumCircuit) -> QuantumCircuit:
    """Rewrite negative RZZ angles for fractional-gate compilation."""
    new = QuantumCircuit(circ.num_qubits, circ.num_clbits)

    for inst in circ.data:
        op = inst.operation
        qargs = inst.qubits
        cargs = inst.clbits

        if op.name == "rzz":
            theta = float(op.params[0])
            if theta < 0:
                abs_theta = -theta
                q0 = qargs[0]
                new.x(q0)
                new.append(RZZGate(abs_theta), qargs)
                new.x(q0)
            else:
                new.append(op, qargs, cargs)
        else:
            new.append(op, qargs, cargs)

    return new


In [ ]:
backend = service.backend("ibm_basquecountry", use_fractional_gates=True)

# Initial logical-to-physical pattern used to avoid SWAP insertion for the
# reservoir and memory-ancilla connectivity.
layout = [10, 11, 18, 31, 30, 29, 12, 32, 28]


In [ ]:
def run_circuit_hw(input_series, circuits, backend, fractional=True, initial_layout=None):
    """Run the circuits on IBM hardware and return expectation values."""
    shots = int(1e4)
    sampler = Sampler(backend)

    # Dynamical decoupling suppresses idle-qubit errors during mid-circuit operations.
    sampler.options.dynamical_decoupling.enable = True
    sampler.options.dynamical_decoupling.sequence_type = "XpXm"

    pm = generate_preset_pass_manager(
        backend=backend,
        optimization_level=3,
        initial_layout=initial_layout,
    )
    transpiled_circuits = pm.run(circuits)

    if fractional:
        transpiled_circuits = [fix_rzz_angles_postprocess(circ) for circ in transpiled_circuits]

    job = sampler.run(transpiled_circuits, shots=shots)
    result = job.result()

    all_expvals = {}
    for obs_idx, obs in enumerate(observables):
        counts = result[obs_idx].data.c.get_counts()
        for k in range(len(input_series)):
            base = k * n_meas_per_step

            lab0 = f"{obs}_0_step_{k}"
            all_expvals[lab0] = exp_z_from_counts(counts, base, shots)

            for j in range(M + R, n_qubits):
                anc_index = base + 1 + (j - (M + R))
                labj = f"{obs}_{j}_step_{k}"
                all_expvals[labj] = exp_z_from_counts(counts, anc_index, shots)

    exp_values = {i: {obs: [] for obs in observables} for i in range(len([0]) + len(list(range(1, M + R, 2))))}

    for obs in observables:
        for k in range(len(input_series)):
            for j in range(M + R, n_qubits):
                labj = f"{obs}_{j}_step_{k}"
                exp_values[j - (M + R) + 1][obs].append(all_expvals[labj] / (2 * alpha**2 - 1))

    for obs in observables:
        for k in range(len(input_series)):
            lab0 = f"{obs}_0_step_{k}"
            exp_values[0][obs].append(all_expvals[lab0])

    return exp_values


## 7. Reservoir parameters and dataset windows

The parameters below match the hardware-tailored setting used for the Santa Fe forecasting data generation. The long input sequence is processed in overlapping windows to respect the practical classical-register limit of the hardware runs.


In [ ]:
M = 5
R = 1
h0 = 1
J = 1
delta_t = 10
g = 2.051
alpha = get_alpha(g)

trotter_reps = 10

# Default workflow: nearest-neighbor hardware-tailored model.
# U_nn_exact is the ideal density-matrix reference for the NN Hamiltonian.
# trotter_step is the gate-level approximation used in the circuit/backend.
U, trotter_step = get_unitary_and_trotter_step(
    M, R, h0, J,
    delta_t=delta_t,
    trotter_reps=trotter_reps,
    seed=60,
)

# Alternative workflow: all-to-all spin-network benchmark, as in Fig. 2.
# This gives an exact dense unitary only; no hardware Trotter step is built here.
# U = get_exact_unitary_all_to_all(M, R, h0, J, delta_t=delta_t, seed=60)

spin_space_M = HilbertSpace(M, "spin")
spin_space_R = HilbertSpace(R, "spin")
composite_space = CompositeHilbertSpace({"R": spin_space_R, "M": spin_space_M})

# Average backaction matrix for the weak indirect measurement.
backaction_matrix = M_matrix(1, g)

observables = ["X", "Y", "Z"]


In [ ]:
n_qubits = R + M + M // 2 + (1 if M % 2 == 1 else 0)
n_meas_per_step = 1 + M // 2 + (1 if M % 2 == 1 else 0)


In [ ]:
K = 1952
dataset = np.load("../sk_Santa_Fe_2000.npy")[:, 0]

# Alternative STM input used in the paper:
# K = 1000
# np.random.seed(0)
# dataset = np.random.uniform(0, 1, K)

K_window = 367
wo_window = 50


In [ ]:
indices = [
    (0, K_window),
    (K_window - wo_window, 2 * K_window - wo_window),
    (2 * K_window - 2 * wo_window, 3 * K_window - 2 * wo_window),
    (3 * K_window - 3 * wo_window, 4 * K_window - 3 * wo_window),
    (4 * K_window - 4 * wo_window, 5 * K_window - 4 * wo_window),
    (5 * K_window - 5 * wo_window, 6 * K_window - 5 * wo_window),
]

indices


## 8. Generate and save data

For each Hamiltonian seed and each window, the notebook computes the ideal reference values, the circuit-simulation values, and the hardware values. Results are written after every stage so that partial hardware runs can be recovered.


In [ ]:
seeds = [60, 19, 34, 46, 47, 48, 90, 121, 136, 149]


In [ ]:
for seed in tqdm(seeds):
    exp_values_FP = {
        "ideal": {},
        "sim": {},
        "hw": {},
    }

    U, trotter_step = get_unitary_and_trotter_step(
        M, R, h0, J,
        delta_t=delta_t,
        trotter_reps=trotter_reps,
        seed=seed,
    )

    for i, j in indices:
        print(i, j, len(dataset[i:j]))
        input_series = dataset[i:j]
        window_id = j // (K_window + 1)

        exp_values_FP["ideal"][window_id] = run_ideal_window(input_series, U)
        print(f"Finished ideal for window {window_id}")
        with open(f"exp_values_FP_basquecountry_{seed}.json", "w") as f:
            json.dump(exp_values_FP, f, indent=4)

        circuits = build_circuit(input_series, trotter_step)

        exp_values_FP["sim"][window_id] = run_circuit_sim(input_series, circuits)
        print(f"Finished sim for window {window_id}")
        with open(f"exp_values_FP_basquecountry_{seed}.json", "w") as f:
            json.dump(exp_values_FP, f, indent=4)

        exp_values_FP["hw"][window_id] = run_circuit_hw(
            input_series,
            circuits,
            backend,
            fractional=True,
            initial_layout=layout,
        )
        print(f"Finished hardware for window {window_id}")

        # Save after each window so partially completed hardware runs are not lost.
        with open(f"exp_values_FP_basquecountry_{seed}_g2.json", "w") as f:
            json.dump(exp_values_FP, f, indent=4)
